In [ ]:
import kagglehub
import pandas as pd
import os
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler,LabelEncoder
from sklearn.model_selection import train_test_split
import numpy as np
from sklearn.model_selection import KFold
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
from catboost import CatBoostRegressor
!pip install catboost

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

full_path = os.path.join(path, 'Q1_data.csv')

In [ ]:
# Task 1: Write your code here:
df = pd.read_csv(full_path)

In [ ]:
# Task 2: Write your code here:
df.head(5)

In [ ]:
# Task 3: Write your code here:
df.info()

In [ ]:
# Task 4: Write your code here:
df.describe()

In [ ]:
df.isna().sum()

In [ ]:
# Task 5: Write your code here:

plt.hist(df['Delivery_Time'].dropna(), bins=50, edgecolor='black');

In [ ]:
# Task 1: Write your code here:
df.drop(columns=['Order_ID'],inplace=True) # Order_ID is irrelevant so drop it


In [ ]:
# Task 2: Write your code here:
df['Weather'].fillna(df['Weather'].mode()[0],inplace=True); #mode
df['Time_of_Day'].fillna(df['Time_of_Day'].mode()[0],inplace=True); #mode
df['Courier_Experience_yrs'].fillna(df['Courier_Experience_yrs'].mode()[0],inplace=True); #mode
df['Traffic_Level'].fillna(df['Traffic_Level'].mode()[0],inplace=True); # mode
df = df.dropna(subset=['Delivery_Time']); # we cannot fill target

In [ ]:
df.isna().sum()

In [ ]:
# Task 3: Write your code here:
duplicates = df.duplicated().sum()  # we have 557 duplicated rows we will drop them all
df = df.drop_duplicates()
print(f'we have: {duplicates} number of duplicate rows')
print(f'Duplicates dropped')


In [ ]:
# Task 4: Write your code here:
df = pd.get_dummies(df, columns=['Weather','Time_of_Day','Vehicle_Type','Traffic_Level'],dtype=int) #one hot encoder as aksed
# recommend some columns such as Traffic_Level, Time_of_Day should be Label encoded becuase depending on trafic and time maybe delivery delay


In [ ]:
# Task 5: Write your code here:
scaler = StandardScaler() # it is wrong to do the scaling on all of the data but i will do as you say.
df[['Distance_km','Preparation_Time_min']] = scaler.fit_transform(df[['Distance_km','Preparation_Time_min']])

In [ ]:
# Task 6: Write your code here:
# ITS A REGRESSION PROBLEM THERE IS SKEWING NOT IMBALANCE

In [ ]:
# Task 1: Write your code here:
X = df.drop(columns=['Delivery_Time']) #Features
y = df['Delivery_Time'] # Target



In [ ]:
# Task 2,3,4,5: Write your code here:
kf = KFold(n_splits=5, shuffle=True, random_state=42) # regression so we use kfold starify only for imbalanced (classification)

metric = []

for fold_idx, (train_index, test_index) in enumerate(kf.split(X)):
  X_train, X_test = X.iloc[train_index], X.iloc[test_index]
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]
  model = RandomForestRegressor(n_estimators=200)
  model.fit(X_train,y_train)
  y_pred = model.predict(X_test)
  mae = mean_absolute_error(y_test,y_pred)
  metric.append(mae)

print(f'Mean MAE: {np.mean(metric)}')

In [ ]:
# Task 1: Write your code here:
pd.Series(model.feature_importances_,index=X.columns).sort_values().plot(kind='barh')
# as i suggested and said before some of these features should be label encoded because they have order to them

In [ ]:
# Task 2: Write your code here:
pr_time = model.predict(X)
plt.plot(pr_time,'o-');

In [ ]:
kf_2 = KFold(n_splits=5, shuffle=True, random_state=42)
rf = []
cb = []
for fold_idx, (train_index, test_index) in enumerate(kf_2.split(X)):

  X_train, X_test = X.iloc[train_index], X.iloc[test_index]
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]

  model_r = RandomForestRegressor(n_estimators=300)
  model_r.fit(X_train,y_train)
  y_pred_r = model.predict(X_test)
  mae_r = mean_absolute_error(y_test,y_pred_r)
  rf.append(mae_r)


  model_c = CatBoostRegressor(verbose=0,n_estimators=200)
  model_c.fit(X_train,y_train)
  y_pred_c = model.predict(X_test)
  mae_c = mean_absolute_error(y_test,y_pred_c)
  cb.append(mae_c)


print(f'RandomForst: {np.mean(rf)}')
print(f'Catboost Forst: {np.mean(cb)}')